# Stride Estimation from IMU Data

This notebook demonstrates how to estimate walking strides and gait parameters from foot-mounted IMU sensor data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/stride_estimation_imu/blob/main/notebooks/demo_colab.ipynb)

## 1. Setup

First, clone the repository and install dependencies.

In [ ]:
# Clone the repository (only needed in Colab)
!git clone https://github.com/jeremydwong/stride_estimation_imu.git
%cd stride_estimation_imu

# Install dependencies
!pip install -q numpy scipy matplotlib h5py

In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
import stride_imu as imu

print("stride_imu loaded successfully!")

## 2. Upload Your Data

Upload your APDM `.h5` sensor file. If you don't have data, you can skip to the synthetic data example below.

In [ ]:
# Upload your .h5 file (run this cell and select your file)
from google.colab import files

print("Select your APDM .h5 file to upload:")
uploaded = files.upload()

# Get the filename
H5_FILE = list(uploaded.keys())[0]
print(f"\nUploaded: {H5_FILE}")

## 3. Load and Process IMU Data

In [ ]:
# Load IMU data
Wb, Ab, PERIOD, _, time_datetime, time_elapsed = imu.getdata_apdm(H5_FILE)

print(f"Loaded {len(Wb)} samples")
print(f"Sampling period: {PERIOD:.6f} s ({1/PERIOD:.1f} Hz)")
print(f"Duration: {len(Wb) * PERIOD:.1f} seconds")

In [ ]:
# Compute position trajectory using inertial mechanization
walk_info = imu.compute_position(Wb, Ab, PERIOD)

print(f"Position trajectory computed: {len(walk_info['P'])} samples")
print(f"Footfalls detected: {np.sum(walk_info['FF'])}")

## 4. Visualize Results

In [ ]:
# Plot 3D trajectory
imu.plt_walk_info_position(walk_info)

In [ ]:
# Segment strides
strides = imu.stride_segmentation(walk_info, PERIOD)

print(f"Strides segmented: {strides['ltrl'].shape[1]} strides")
print(f"Mean forward distance: {np.mean(strides['frwd'][-1, :]):.3f} m")
print(f"Mean stride speed: {np.mean(strides['frwd_speed']):.3f} m/s")

In [ ]:
# Plot lateral vs forward stride trajectories
imu.plt_ltrl_frwd_strides(strides)

In [ ]:
# Plot stride variability ellipse
imu.plt_stride_var(strides)

In [ ]:
# Plot forward vs elevation
imu.plt_frwd_elev_strides(strides)

## 5. Gait Parameters Summary

In [ ]:
# Compute summary statistics
step_lengths = strides['frwd'][-1, :]
step_speeds = strides['frwd_speed']
step_times = strides['time']

print("=" * 40)
print("GAIT PARAMETERS SUMMARY")
print("=" * 40)
print(f"Number of strides: {len(step_lengths)}")
print(f"")
print(f"Step Length:")
print(f"  Mean: {np.mean(step_lengths):.3f} m")
print(f"  Std:  {np.std(step_lengths):.3f} m")
print(f"  Range: {np.min(step_lengths):.3f} - {np.max(step_lengths):.3f} m")
print(f"")
print(f"Step Speed:")
print(f"  Mean: {np.mean(step_speeds):.3f} m/s")
print(f"  Std:  {np.std(step_speeds):.3f} m/s")
print(f"")
print(f"Step Duration:")
print(f"  Mean: {np.mean(step_times):.3f} s")
print(f"  Std:  {np.std(step_times):.3f} s")
print("=" * 40)

In [ ]:
# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(step_lengths, bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Step Length [m]')
axes[0].set_ylabel('Count')
axes[0].set_title('Step Length Distribution')
axes[0].axvline(np.mean(step_lengths), color='r', linestyle='--', label=f'Mean: {np.mean(step_lengths):.3f}')
axes[0].legend()

axes[1].hist(step_speeds, bins=20, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Step Speed [m/s]')
axes[1].set_ylabel('Count')
axes[1].set_title('Step Speed Distribution')
axes[1].axvline(np.mean(step_speeds), color='r', linestyle='--', label=f'Mean: {np.mean(step_speeds):.3f}')
axes[1].legend()

axes[2].hist(step_times, bins=20, edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Step Duration [s]')
axes[2].set_ylabel('Count')
axes[2].set_title('Step Duration Distribution')
axes[2].axvline(np.mean(step_times), color='r', linestyle='--', label=f'Mean: {np.mean(step_times):.3f}')
axes[2].legend()

plt.tight_layout()
plt.show()